In [144]:
import warnings
warnings.filterwarnings("ignore")

In [145]:
import datasets
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf

In [146]:
from datasets import load_dataset
import pandas as pd
ds=load_dataset("Tobi-Bueck/customer-support-tickets")

In [147]:
df=pd.DataFrame(ds['train'])
df.to_csv("data.csv")

KeyboardInterrupt: 

In [ ]:
df=df[['body','queue','language']]
df.head()

,body,queue,language
0,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Technical Support,de
1,"Dear Customer Support Team,\n\nI am writing to...",Technical Support,en
2,"Dear Customer Support Team,\n\nI hope this mes...",Returns and Exchanges,en
3,"Dear Customer Support Team,\n\nI hope this mes...",Billing and Payments,en
4,"Dear Support Team,\n\nI hope this message reac...",Sales and Pre-Sales,en


In [ ]:
df['body'].isnull().sum()
df['body']=df['body'].fillna("")
df['body']=df['body'].fillna("").str.lower()

In [ ]:
import re

df['body']=df['body'].apply(lambda x: re.sub(r'nn+',' ',x))
df['body']=df['body'].apply(lambda x: re.sub(r'\s+', ' ',x).strip())
df['body']=df['body'].apply(lambda x: re.sub(r'([a-z])([A-Z])',r'\1 \2',x))

In [ ]:
import re
import string

df['body']=df['body'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

In [ ]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from nltk.tokenize import word_tokenize
df['tokens']=df['body'].apply(word_tokenize)
df.head()

,body,queue,language,tokens
0,sehr geehrtes supportteamnnich möchte einen gr...,Technical Support,de,"[sehr, geehrtes, supportteamnnich, möchte, ein..."
1,dear customer support teamnni am writing to re...,Technical Support,en,"[dear, customer, support, teamnni, am, writing..."
2,dear customer support teamnni hope this messag...,Returns and Exchanges,en,"[dear, customer, support, teamnni, hope, this,..."
3,dear customer support teamnni hope this messag...,Billing and Payments,en,"[dear, customer, support, teamnni, hope, this,..."
4,dear support teamnni hope this message reaches...,Sales and Pre-Sales,en,"[dear, support, teamnni, hope, this, message, ..."


In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

english=set(stopwords.words('english'))
german=set(stopwords.words('german'))

df['tokens']=df['tokens'].apply(
    lambda words:[w.lower() for w in words if w.isalpha()]
)
df.loc[df['language'] == 'en', 'tokens'] = df.loc[df['language'] == 'en', 'tokens'].apply(
    lambda words: [w for w in words if w not in english]
)

df.loc[df['language'] == 'de', 'tokens'] = df.loc[df['language'] == 'de', 'tokens'].apply(
    lambda words: [w for w in words if w not in german]
)

df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,body,queue,language,tokens
0,sehr geehrtes supportteamnnich möchte einen gr...,Technical Support,de,"[geehrtes, supportteamnnich, möchte, gravieren..."
1,dear customer support teamnni am writing to re...,Technical Support,en,"[dear, customer, support, teamnni, writing, re..."
2,dear customer support teamnni hope this messag...,Returns and Exchanges,en,"[dear, customer, support, teamnni, hope, messa..."
3,dear customer support teamnni hope this messag...,Billing and Payments,en,"[dear, customer, support, teamnni, hope, messa..."
4,dear support teamnni hope this message reaches...,Sales and Pre-Sales,en,"[dear, support, teamnni, hope, message, reache..."


In [ ]:
df['tokens']=df['tokens'].apply(lambda words:[w.lower() for w in words])
df.head()

,body,queue,language,tokens
0,sehr geehrtes supportteamnnich möchte einen gr...,Technical Support,de,"[geehrtes, supportteamnnich, möchte, gravieren..."
1,dear customer support teamnni am writing to re...,Technical Support,en,"[dear, customer, support, teamnni, writing, re..."
2,dear customer support teamnni hope this messag...,Returns and Exchanges,en,"[dear, customer, support, teamnni, hope, messa..."
3,dear customer support teamnni hope this messag...,Billing and Payments,en,"[dear, customer, support, teamnni, hope, messa..."
4,dear support teamnni hope this message reaches...,Sales and Pre-Sales,en,"[dear, support, teamnni, hope, message, reache..."


In [ ]:
df['tokens']=df['tokens'].apply(lambda words: [w for w in words if w.isalpha()])
df['tokens'].head()

0    [geehrtes, supportteamnnich, möchte, gravieren...
1    [dear, customer, support, teamnni, writing, re...
2    [dear, customer, support, teamnni, hope, messa...
3    [dear, customer, support, teamnni, hope, messa...
4    [dear, support, teamnni, hope, message, reache...
Name: tokens, dtype: object

In [ ]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
from nltk.stem import WordNetLemmatizer

lemma=WordNetLemmatizer()
df['tokens']=df['tokens'].apply(
    lambda words:[lemma.lemmatize(w) for w in words]
)
df.head()

,body,queue,language,tokens
0,sehr geehrtes supportteamnnich möchte einen gr...,Technical Support,de,"[geehrtes, supportteamnnich, möchte, gravieren..."
1,dear customer support teamnni am writing to re...,Technical Support,en,"[dear, customer, support, teamnni, writing, re..."
2,dear customer support teamnni hope this messag...,Returns and Exchanges,en,"[dear, customer, support, teamnni, hope, messa..."
3,dear customer support teamnni hope this messag...,Billing and Payments,en,"[dear, customer, support, teamnni, hope, messa..."
4,dear support teamnni hope this message reaches...,Sales and Pre-Sales,en,"[dear, support, teamnni, hope, message, reach,..."


In [ ]:
df['tokens']=df['tokens'].apply(lambda words: " ".join(words))

In [ ]:
print(df.columns)

Index(['body', 'queue', 'language', 'tokens'], dtype='object')


In [ ]:
x=df['tokens']
y=df['queue']

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(max_features=5000)
x_train_vec=tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [ ]:
df.to_csv("clean_data.csv")

In [ ]:
df = ds['train'].to_pandas()

In [ ]:
from sklearn.linear_model import LogisticRegression
m1=LogisticRegression(max_iter=1000,class_weight='balanced')
m1.fit(x_train_vec,y_train)
y_pred=m1.predict(X_test_vec)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.7173965838257913
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.98      0.78      0.87        54
            Arts & Entertainment/Music       0.87      0.71      0.79        77
          Autos & Vehicles/Maintenance       0.67      0.76      0.71        62
                Autos & Vehicles/Sales       0.65      0.70      0.67        66
            Beauty & Fitness/Cosmetics       0.80      0.85      0.82        55
     Beauty & Fitness/Fitness Training       0.88      0.70      0.78        60
                  Billing and Payments       0.94      0.78      0.85       990
            Books & Literature/Fiction       0.64      0.75      0.69        52
        Books & Literature/Non-Fiction       0.80      0.78      0.79        63
   Business & Industrial/Manufacturing       0.75      0.74      0.74        73
                      Customer Service       0.67      0.67      0

In [ ]:
from sklearn.naive_bayes import MultinomialNB

m2=MultinomialNB()
m2.fit(x_train_vec,y_train)
y_pred=m2.predict(X_test_vec)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.434955071642516
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.95      0.37      0.53        54
            Arts & Entertainment/Music       1.00      0.27      0.43        77
          Autos & Vehicles/Maintenance       0.77      0.27      0.40        62
                Autos & Vehicles/Sales       0.39      0.68      0.49        66
            Beauty & Fitness/Cosmetics       0.73      0.55      0.62        55
     Beauty & Fitness/Fitness Training       0.93      0.42      0.57        60
                  Billing and Payments       0.88      0.56      0.69       990
            Books & Literature/Fiction       0.72      0.50      0.59        52
        Books & Literature/Non-Fiction       0.52      0.67      0.58        63
   Business & Industrial/Manufacturing       0.62      0.71      0.66        73
                      Customer Service       0.28      0.45      0.

In [ ]:
from sklearn.svm import LinearSVC

m3=LinearSVC()
m3.fit(x_train_vec,y_train)
y_pred=m3.predict(X_test_vec)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.5465069213956124
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.85      0.76      0.80        54
            Arts & Entertainment/Music       0.76      0.75      0.76        77
          Autos & Vehicles/Maintenance       0.58      0.58      0.58        62
                Autos & Vehicles/Sales       0.62      0.67      0.64        66
            Beauty & Fitness/Cosmetics       0.77      0.80      0.79        55
     Beauty & Fitness/Fitness Training       0.81      0.63      0.71        60
                  Billing and Payments       0.77      0.71      0.74       990
            Books & Literature/Fiction       0.75      0.69      0.72        52
        Books & Literature/Non-Fiction       0.75      0.81      0.78        63
   Business & Industrial/Manufacturing       0.85      0.78      0.81        73
                      Customer Service       0.40      0.40      0

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

m4=RandomForestClassifier(class_weight=class_weights)
m4.fit(x_train_vec,y_train)
y_pred=m4.predict(X_test_vec)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.7173965838257913
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.98      0.78      0.87        54
            Arts & Entertainment/Music       0.87      0.71      0.79        77
          Autos & Vehicles/Maintenance       0.67      0.76      0.71        62
                Autos & Vehicles/Sales       0.65      0.70      0.67        66
            Beauty & Fitness/Cosmetics       0.80      0.85      0.82        55
     Beauty & Fitness/Fitness Training       0.88      0.70      0.78        60
                  Billing and Payments       0.94      0.78      0.85       990
            Books & Literature/Fiction       0.64      0.75      0.69        52
        Books & Literature/Non-Fiction       0.80      0.78      0.79        63
   Business & Industrial/Manufacturing       0.75      0.74      0.74        73
                      Customer Service       0.67      0.67      0

In [ ]:
import pickle

with open("model1.pkl","wb") as f:
  pickle.dump((m4,tfidf),f)

In [169]:
text=df['tokens']

In [172]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
tokenizer = Tokenizer(num_words=20000,oov_token="<OOV>")
tokenizer.fit_on_texts(text)
sequences = tokenizer.texts_to_sequences(text)
pad=pad_sequences(sequences,maxlen=150,padding='post',truncating='post')

from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
y=le.fit_transform(df['queue'])

In [173]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    pad,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional,Dropout
from tensorflow.keras.optimizers import Adam # Import Adam optimizer

num_classes=len(set(y))
model=Sequential()

# Set input_length to 50 to match maxlen used in pad_sequences
model.add(Embedding(input_dim=5000+1,output_dim=150,input_length=60))
model.add(Bidirectional(LSTM(128, return_sequences=False)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dense(num_classes,activation='softmax'))

# Use 'sparse_categorical_crossentropy' with integer labels
model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=0.001),metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=1,
    min_lr=1e-5
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_lstm),
    y=y_train_lstm
)

class_weights = dict(zip(np.unique(y_train_lstm), class_weights))


In [ ]:
model.fit(
    x_train_lstm,
    y_train_lstm,
    epochs=10,
    batch_size=16,
    validation_data=(x_test_lstm, y_test_lstm),
    callbacks=[early_stop, lr_reduce],
    # class_weight=class_weights
)

Epoch 1/10
 557/3089 ━━━━━━━━━━━━━━━━━━━━ 33s 13ms/step - accuracy: 0.2009 - loss: 2.6383

In [ ]:
model.save("lstm_model.keras")

In [ ]:
import pickle

with open("lstm.pkl","wb") as f:
  pickle.dump(tokenizer, f)

In [ ]:
model.summary()

NameError: name 'model' is not defined

In [ ]:
from sklearn.metrics import accuracy_score,classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
df['tokens'] = df['tokens'].apply(lambda x: x.split())

In [ ]:
print(df['tokens'].head())

0    [geehrtes, supportteamnnich, möchte, gravieren...
1    [dear, customer, support, teamnni, writing, re...
2    [dear, customer, support, teamnni, hope, messa...
3    [dear, customer, support, teamnni, hope, messa...
4    [dear, support, teamnni, hope, message, reach,...
Name: tokens, dtype: object


In [ ]:
unique_words = set()

for tokens in df['tokens']:
    unique_words.update(tokens)

print("Total Unique Words:", len(unique_words))

Total Unique Words: 48836


In [174]:
from collections import Counter

counter = Counter()

for tokens in df['tokens']:
    counter.update(tokens)

# Vocabulary dictionary
vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}

# Add words to vocab
for word, freq in counter.items():

    # Keep words appearing at least 2 times
    if freq >= 2:
        vocab[word] = len(vocab)

vocab_size = len(vocab)

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 28204


In [ ]:
!pip install datasets nltk scikit-learn torch

  Using cached torch-2.11.0-cp310-cp310-win_amd64.whl (114.5 MB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
     ---------------------------------------- 0.0/536.2 kB ? eta -:--:--
      --------------------------------------- 10.2/536.2 kB ? eta -:--:--
     -- ---------------------------------- 41.0/536.2 kB 393.8 kB/s eta 0:00:02
     ---- -------------------------------- 71.7/536.2 kB 438.9 kB/s eta 0:00:02
     ------ ----------------------------- 102.4/536.2 kB 535.8 kB/s eta 0:00:01
     --------- -------------------------- 143.4/536.2 kB 568.9 kB/s eta 0:00:01
     --------- -------------------------- 143.4/536.2 kB 568.9 kB/s eta 0:00:01
     ---------- ------------------------- 153.6/536.2 kB 482.7 kB/s eta 0:00:01
     ----------- ------------------------ 174.1/536.2 kB 476.3 kB/s eta 0:00:01
     ------------ ----------------------- 184.3/536.2 kB 446.4 kB/s eta 0:00:01
     ------------- -------------------


[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from collections import Counter

In [175]:
class LSTMClassifier(nn.Module):

    def __init__(self,
                 vocab_size,
                 embedding_dim,
                 hidden_dim,
                 output_dim,
                 num_layers=2,
                 dropout=0.3):

        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        embedded = self.embedding(x)

        lstm_out, (hidden, cell) = self.lstm(embedded)

        # Concatenate forward + backward hidden states
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)

        hidden = self.dropout(hidden)

        output = self.fc(hidden)

        return output

In [161]:
from torch.utils.data import Dataset, DataLoader
import torch

class TicketDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

In [176]:
train_dataset = TicketDataset(X_train, y_train)

test_dataset = TicketDataset(X_test, y_test)

In [177]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

In [ ]:
num_classes = len(le.classes_)

print(num_classes)

52


In [178]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes=52

model = LSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=128,
    output_dim=num_classes
)

model = model.to(device)

print(model)

LSTMClassifier(
  (embedding): Embedding(28204, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=256, out_features=52, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [179]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to tensor
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

# Loss function with class weights
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 18

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for texts, labels in train_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

Epoch [1/18], Loss: 2.5668
Epoch [2/18], Loss: 1.9036
Epoch [3/18], Loss: 1.3605


In [167]:
model.eval()

predictions = []
actuals = []

with torch.no_grad():

    for texts, labels in test_loader:

        texts = texts.to(device)

        outputs = model(texts)

        _, preds = torch.max(outputs, 1)

        predictions.extend(preds.cpu().numpy())

        actuals.extend(labels.numpy())

In [168]:
accuracy = accuracy_score(actuals, predictions)

print("Accuracy:", accuracy)

print(classification_report(actuals, predictions))

Accuracy: 0.5010928519388003
              precision    recall  f1-score   support

           0       0.32      0.44      0.37        54
           1       0.15      0.16      0.15        77
           2       0.13      0.18      0.15        62
           3       0.18      0.29      0.22        66
           4       0.29      0.25      0.27        55
           5       0.20      0.22      0.21        60
           6       0.85      0.72      0.78       990
           7       0.28      0.46      0.35        52
           8       0.48      0.32      0.38        63
           9       0.25      0.40      0.31        73
          10       0.50      0.49      0.50      1460
          11       0.29      0.23      0.25        66
          12       0.56      0.61      0.58        54
          13       0.18      0.18      0.18        71
          14       0.14      0.13      0.14        60
          15       0.29      0.13      0.18        67
          16       0.55      0.26      0.35       12